# Notebook 8 – Advanced Boosting Algorithms

**Dataset:** Online Retail transactions (`data.csv`)

**Task:** Predict whether an order is from the **United Kingdom** or not, using `Quantity`, `UnitPrice`, `TotalPrice`.

We cover **XGBoost**, **LightGBM**, and **CatBoost** — the 3 most popular advanced boosting libraries.

## Installation

These are external libraries (not part of scikit-learn) and need to be installed separately:

```bash
pip install xgboost lightgbm catboost
```

No special configuration is needed beyond installing the package — each provides a scikit-learn-compatible API (`.fit()`, `.predict()`), so they slot into a normal workflow easily.

In [7]:
%pip install xgboost lightgbm catboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\hemak\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [8]:
import xgboost
import lightgbm
import catboost
print("xgboost:", xgboost.__version__)
print("lightgbm:", lightgbm.__version__)
print("catboost:", catboost.__version__)

xgboost: 3.4.1
lightgbm: 4.7.0
catboost: 1.2.10


## Setup: Load & Split Data

In [9]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].sample(3000, random_state=42)
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
X = df[['Quantity', 'UnitPrice', 'TotalPrice']]
y = (df['Country'] == 'United Kingdom').astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 1. XGBoost (Extreme Gradient Boosting)

**What it is:** A highly optimized, regularized implementation of Gradient Boosting, one of the most widely used ML libraries in competitions and industry.

**How it works:** Builds trees sequentially like standard Gradient Boosting, but adds L1/L2 regularization on leaf weights, uses a more precise loss approximation (second-order gradients), and includes built-in handling for missing values.

**Important hyperparameters:**
- `n_estimators` – number of boosting rounds (trees)
- `max_depth` – depth of each tree
- `learning_rate` – how much each tree contributes (shrinkage)
- `subsample` – fraction of rows used per tree (adds randomness)
- `colsample_bytree` – fraction of features used per tree

**Advantages:** very accurate, built-in regularization reduces overfitting, handles missing values natively, fast (parallelized).
**Limitations:** more hyperparameters to tune than simpler models; can still overfit on small/noisy datasets; less interpretable than a single tree.
**When to use it:** structured/tabular data, competitions, when top accuracy matters and you have time to tune.
**When not to use it:** very small datasets, when interpretability is critical, or when a simpler model already performs well enough.

In [10]:
from xgboost import XGBClassifier
start = time.time()
xgb_model = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train, y_train)
xgb_time = time.time() - start
print("XGBoost test acc:", accuracy_score(y_test, xgb_model.predict(X_test)))
print("Training time (s):", round(xgb_time, 3))

XGBoost test acc: 0.8916666666666667
Training time (s): 0.581


## 2. LightGBM (Light Gradient Boosting Machine)

**What it is:** A gradient boosting framework by Microsoft, designed to be very fast and memory-efficient, especially on large datasets.

**How it works:** Uses histogram-based binning of continuous features (like sklearn's HistGradientBoosting) and grows trees **leaf-wise** (splitting the leaf that reduces loss the most) rather than level-wise, making it faster and often more accurate per tree.

**Important hyperparameters:**
- `n_estimators` – number of boosting rounds
- `num_leaves` – max leaves per tree (controls complexity directly)
- `learning_rate` – shrinkage per tree
- `max_depth` – optional depth limit (leaf-wise growth can get deep fast)
- `min_child_samples` – minimum data needed in a leaf

**Advantages:** very fast training, low memory use, handles large datasets well, good accuracy.
**Limitations:** leaf-wise growth can overfit on small datasets if not tuned carefully (e.g., `num_leaves` too high); more sensitive to hyperparameters than XGBoost on small data.
**When to use it:** large datasets, when training speed matters.
**When not to use it:** very small datasets (can overfit quickly without careful tuning).

In [ ]:
from lightgbm import LGBMClassifier
start = time.time()
lgbm_model = LGBMClassifier(n_estimators=100, num_leaves=15, learning_rate=0.1, random_state=42, verbose=-1)
lgbm_model.fit(X_train, y_train)
lgbm_time = time.time() - start
print("LightGBM test acc:", accuracy_score(y_test, lgbm_model.predict(X_test)))
print("Training time (s):", round(lgbm_time, 3))

## 3. CatBoost (Categorical Boosting)

**What it is:** A gradient boosting library by Yandex, built with **native, automatic handling of categorical features** as a core strength.

**How it works:** Uses "ordered boosting" (a technique to reduce prediction bias/target leakage) and encodes categorical features internally using statistics computed in a way that avoids leaking target information — no manual one-hot/label encoding needed.

**Important hyperparameters:**
- `iterations` – number of boosting rounds (like `n_estimators`)
- `depth` – depth of each tree
- `learning_rate` – shrinkage per tree
- `cat_features` – which columns to treat as categorical
- `l2_leaf_reg` – L2 regularization strength

**Advantages:** excellent with categorical features out of the box, strong default hyperparameters (often works well with minimal tuning), robust to overfitting via ordered boosting.
**Limitations:** can be slower to train than LightGBM on very large datasets; larger model files; smaller community/ecosystem than XGBoost.
**When to use it:** datasets with many categorical features (e.g., `Country`, `StockCode`, `Description` in our data).
**When not to use it:** purely numeric datasets with no categorical columns, where the main benefit doesn't apply — a simpler booster may be just as good and faster.

In [11]:
from catboost import CatBoostClassifier
start = time.time()
cat_model = CatBoostClassifier(iterations=100, depth=4, learning_rate=0.1, random_state=42, verbose=0)
cat_model.fit(X_train, y_train)
cat_time = time.time() - start
print("CatBoost test acc:", accuracy_score(y_test, cat_model.predict(X_test)))
print("Training time (s):", round(cat_time, 3))

CatBoost test acc: 0.8983333333333333
Training time (s): 0.582


## Bonus: CatBoost with a Real Categorical Feature
Since CatBoost's main strength is categorical features, let's actually give it one (`Country` itself would leak the answer, so we use `StockCode` instead as an example categorical feature).

In [12]:
df2 = df.copy()
df2['StockCode'] = df2['StockCode'].astype(str)
X_cat = df2[['Quantity', 'UnitPrice', 'StockCode']]
y_cat = (df2['Country'] == 'United Kingdom').astype(int)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_cat, y_cat, test_size=0.2, random_state=42, stratify=y_cat)
cat_model2 = CatBoostClassifier(iterations=100, depth=4, learning_rate=0.1, random_state=42, verbose=0)
cat_model2.fit(Xc_train, yc_train, cat_features=['StockCode'])
print("CatBoost with categorical feature — Test acc:", accuracy_score(yc_test, cat_model2.predict(Xc_test)))

CatBoost with categorical feature — Test acc: 0.9016666666666666


## Comparison Summary

In [ ]:
summary = pd.DataFrame({
    'Library': ['XGBoost', 'LightGBM', 'CatBoost'],
    'Test Accuracy': [
        accuracy_score(y_test, xgb_model.predict(X_test)),
        accuracy_score(y_test, lgbm_model.predict(X_test)),
        accuracy_score(y_test, cat_model.predict(X_test)),
    ],
    'Training Time (s)': [round(xgb_time, 3), round(lgbm_time, 3), round(cat_time, 3)],
    'Best For': ['General purpose, competitions', 'Large datasets, speed', 'Categorical-heavy data']
})
summary